In [10]:
!hostname

In [11]:
import multiprocessing
import json
import numpy as np
import pathlib as Path
import difflib

benchmark_names = [f.stem for f in Path.Path('../trace/sequences').glob('*.json')]
# sort the benchmark names
benchmark_names.sort()
benchmarks = []
for benchmark_name in benchmark_names:
    print(f'Processing {benchmark_name}')
    sequences = []
    with open(f'../trace/sequences/{benchmark_name}.json') as f:
        for line in f:
            sequence = json.loads(line)
            if sequence[-1] != '<_fini>':
                print(f'[{benchmark_name}] expected <_fini>: ')
                print(sequence[-10:])
            sequences.append(sequence)
    print(f'Loaded {len(sequences)} sequences')


    # Step 1: Define the function to be parallelized
    def calculate_similarity(pair):
        i, j = pair
        return difflib.SequenceMatcher(None, sequences[i], sequences[j]).ratio()


    similarity_matrix = np.zeros((len(sequences), len(sequences)))

    # Step 2: Create a multiprocessing Pool
    with multiprocessing.Pool() as pool:
        # Generate all pairs of indices (not repeated)
        pairs = [(i, j) for i in range(len(sequences)) for j in range(i + 1, len(sequences))]
        print(f'Calculating {len(pairs)} pairs')

        # Step 3: Use imap to apply the function to each pair
        results = pool.imap(calculate_similarity, pairs)

        # Store results in the similarity matrix
        for pair, result in zip(pairs, results):
            i, j = pair
            similarity_matrix[i, j] = result

    similarity_scores = similarity_matrix[np.triu_indices(len(sequences), k=1)]
    benchmarks.append((sequences, similarity_scores))

In [12]:
import matplotlib.pyplot as plt
import numpy as np

# for each benchmark plot a row with 2 cols, first one is the violin plot of sequences lens, and the other one is the violin plot of similarity_scores
fig, axs = plt.subplots(len(benchmarks), 2, figsize=(10, 1 * len(benchmarks)), dpi=320)

for idx, (seqs, scores) in enumerate(benchmarks):
    lengths = [len(seq) for seq in seqs]

    # draw lens
    axs[idx, 0].violinplot(lengths, vert=False, showmeans=True, showextrema=True)
    axs[idx, 0].set_xlabel(f'{len(seqs)} sequences')
    axs[idx, 0].set_yticks([])
    # lim is 0 to 10000
    axs[idx, 0].set_xlim(0, 10000)
    # print in k format without decimal points
    axs[idx, 0].set_xticks(np.arange(0, 10000 + 1, 1000))
    axs[idx, 0].set_xticklabels([f'{i // 1000}K' for i in np.arange(0, 10000 + 1, 1000)])

    # draw a violin plot
    axs[idx, 1].violinplot(scores, vert=False, showmeans=True, showextrema=True)
    axs[idx, 1].set_xlabel(f'{len(scores)} pairs')
    axs[idx, 1].set_yticks([])
    # xticks every .1
    axs[idx, 1].set_xticks(np.arange(0, 1.1, 0.1))
    # lim is 0 to 1
    axs[idx, 1].set_xlim(0, 1)
    # remove top left and right spines
    for ax in axs[idx]:
        # ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)

    axs[idx, 0].set_title(f'{benchmark_names[idx]} Sequence Lengths')
    axs[idx, 1].set_title(f'{benchmark_names[idx]} Similarity Scores')

plt.tight_layout()

# to pdf
plt.savefig('out/control-flow-variation.pdf', bbox_inches='tight', pad_inches=0, dpi=320, format='pdf',
            transparent=True)

plt.show()

In [13]:
# generate a json for the benchmark names, number of workloads, number of pairs, length of the sequences (min, max, mean), similarity score (min, max, mean)
import json
import numpy as np

data = []
for (seqs, scores), benchmark_name in zip(benchmarks, benchmark_names):
    # remove _r from the benchmark name
    if benchmark_name.endswith('_r'):
        benchmark_name = benchmark_name[:-2]
    # if starts with numbers and . remove the number and the dot
    while benchmark_name[0].isdigit() or benchmark_name[0] == '.':
        benchmark_name = benchmark_name[1:]

    data.append({
        'benchmark_name': benchmark_name,
        'num_workloads': len(seqs),
        'num_pairs': len(scores),
        'sequence_length': {
            'min': min(len(seq) for seq in seqs),
            'max': max(len(seq) for seq in seqs),
            'mean': np.mean([len(seq) for seq in seqs], dtype=int)
        },
        'similarity_score': {
            'min': round(min(scores), 2),
            'max': round(max(scores), 2),
            'mean': round(np.mean(scores), 2)
        }
    })

data

In [15]:
# generate a latex table from the data using this template:
# \begin{table}[ht]
# \centering
# \renewcommand{\arraystretch}{.8} % Makes the table more compact!
# \begin{tabular}{lcccccccc}
# \toprule
# \multirow{2}{*}{\textbf{Benchmark}} & \multirow{2}{*}{\textbf{\#Workloads}} & \multirow{2}{*}{\parbox{1.5cm}{\centering \textbf{Avg.\\Length}}} & \multicolumn{3}{c}{\textbf{Similarity Score}} \\
# \cmidrule(lr){4-6}
# & & & \textbf{Min} & \textbf{Max} & \textbf{Avg} \\
# \midrule
# gcc & 162 & 7668 & 0.33 & 1.0 & 0.62 \\
# mcf & 3 & 61 & 1.0 & 1.0 & 1.0 \\
# cactuBSSN & 7 & 907 & 1.0 & 1.0 & 1.0 \\
# povray & 7 & 506 & 0.71 & 0.97 & 0.82 \\
# lbm & 24 & 29 & 0.93 & 1.0 & 0.97 \\
# xalancbmk & 5 & 1665 & 0.84 & 0.92 & 0.88 \\
# deepsjeng & 9 & 100 & 0.77 & 0.92 & 0.86 \\
# leela & 9 & 216 & 0.92 & 0.99 & 0.96 \\
# nab & 8 & 97 & 1.0 & 1.0 & 1.0 \\
# exchange2 & 10 & 42 & 0.98 & 1.0 & 0.99 \\
# xz & 32 & 149 & 0.25 & 1.0 & 0.9 \\
# git & 8 & 501 & 0.26 & 0.8 & 0.5 \\
# sqlite3 & 4 & 498 & 0.72 & 0.98 & 0.85 \\
# x264 & 13 & 491 & 0.51 & 0.96 & 0.75 \\
# \bottomrule
# \end{tabular}
# \todo[inline, color=blue!20!white]{Amir: I ran all the alberta workloads except 3 of them that I couldn't compile their binary or I got runtime errors. I also have 'git' and 'sqlite3', if you're interested.}
# \caption{Similarity scores for JIT compilation sequences for the SPEC CPU2017 benchmarks across the Alberta workloads. A similarity score of 1.0 means that the sequences are identical, and 0.0 if they have nothing in common. We notice that even for these complex applications, the sequences of JIT compiled functions when run with different input data sets are highly similar.}
# \label{table:similarity_scores}
# \end{table}

def generate_latex_table(data):
    table = r"""
\begin{table}[ht]
\centering
\renewcommand{\arraystretch}{.8} % Makes the table more compact!
\begin{tabular}{lcccccccc}
\toprule
\multirow{2}{*}{\textbf{Benchmark}} & \multirow{2}{*}{\textbf{\#Workloads}} & \multirow{2}{*}{\parbox{1cm}{\centering \textbf{Avg.\\Length}}} & \multicolumn{3}{c}{\textbf{Similarity Score}} \\
\cmidrule(lr){4-6}
& & & \textbf{Min} & \textbf{Max} & \textbf{Avg} \\
\midrule
"""
    for d in data:
        table += f"{d['benchmark_name']} & {d['num_workloads']} & {d['sequence_length']['mean']} & {d['similarity_score']['min']} & {d['similarity_score']['max']} & {d['similarity_score']['mean']} \\\\\n"
    table += r"""
\bottomrule
\end{tabular}
\todo[inline, color=blue!20!white]{Amir: I ran all the alberta workloads except 3 of them (510.parest, 520.omnetpp, 526.blender) that I couldn't compile their binary or I got runtime errors. I also have 'git' and 'sqlite3', if you're interested.}
\caption{Similarity scores for JIT compilation sequences for the SPEC CPU2017 benchmarks across the Alberta workloads. A similarity score of 1.0 means that the sequences are identical, and 0.0 if they have nothing in common. We notice that even for these complex applications, the sequences of JIT compiled functions when run with different input data sets are highly similar.}
\label{table:similarity_scores}
\end{table}
"""
    return table


print(generate_latex_table(data))

In [54]:
def clusterify(similarity_threshold=0.6):
    clusters = [[i] for i in range(len(sequences))]
    while True:
        max_similarity = 0
        max_i = 0
        max_j = 0
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                similarity = 0
                for seq_i in clusters[i]:
                    for seq_j in clusters[j]:
                        similarity += similarity_matrix[seq_i, seq_j]
                similarity /= len(clusters[i]) * len(clusters[j])
                if similarity > max_similarity:
                    max_similarity = similarity
                    max_i = i
                    max_j = j
        if max_similarity < similarity_threshold:
            break
        clusters[max_i].extend(clusters[max_j])
        clusters.pop(max_j)
    return clusters

In [56]:
for similarity_trheshold in np.arange(0.5, .9, 0.1):
    clusters = clusterify(similarity_trheshold)
    print(f'Similarity Threshold {similarity_trheshold:.1f} ==> {len(clusters)} clusters')
    for idx, cluster in enumerate(clusters):
        print(f'{idx + 1}: {cluster}')
    print()